# ADNI Original Volume and Shape Speeds

This notebook uses the original left hippocampus meshes only.

- Step 1: no-MCI original dataset, CN vs AD volume-change speeds.
- Step 2: with-MCI original dataset, CN vs MCI vs AD volume-change speeds.
- Step 3: local shape-speed visualizations for original meshes.

The speed computation uses adjacent longitudinal visits and converts each change to a yearly rate using `delta_months / 12`.

The notebook now includes the common longitudinal figure set used in progression papers:

- all-subject spaghetti trajectories;
- cohort mean trajectories with 95% confidence bands;
- baseline-normalized change trajectories;
- individual interval speed trajectories;
- cohort average speed trajectories;
- subject mean speed distributions and age-speed summaries.

In [14]:
from pathlib import Path
import sys
from importlib import reload

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'examples').is_dir():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

repo_root = find_repo_root(Path.cwd())
helper_dir = repo_root / 'examples' / 'ADNI_1_L_With_MCI'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import adni_original_speed_helpers as helpers
helpers = reload(helpers)

no_mci_root = repo_root / 'examples' / 'ADNI_1_L_No_MCI' / 'brainode_comparison_task1_manifest_original'
with_mci_root = repo_root / 'examples' / 'ADNI_1_L_With_MCI' / 'brainode_comparison_task1_manifest_original'

no_mci = helpers.load_manifest(
    no_mci_root,
    helpers.NO_MCI_PREFIX,
    name='ADNI original no-MCI',
)
with_mci = helpers.load_manifest(
    with_mci_root,
    helpers.WITH_MCI_PREFIX,
    name='ADNI original with-MCI',
)


## Step 1

Original no-MCI dataset only. These figures compare normal aging (`CN`) and disease progression (`AD`) using adjacent-visit volume-change speeds, all-subject trajectories, and cohort-average longitudinal summaries.

In [15]:
for fig in helpers.step1_figures(no_mci):
    fig.show()

for fig in helpers.common_longitudinal_figures(no_mci, ['CN', 'AD'], 'no-MCI'):
    fig.show()


## Step 2

Original with-MCI dataset. These figures compare `CN`, `MCI`, and `AD` using the same adjacent-visit yearly speed conversion, all-subject trajectories, and cohort-average longitudinal summaries.

In [16]:
for fig in helpers.step2_figures(with_mci):
    fig.show()

for fig in helpers.common_longitudinal_figures(with_mci, ['CN', 'MCI', 'AD'], 'with-MCI'):
    fig.show()


## Step 3

Local shape-speed maps on the original meshes.

- Signed normal speed: outward positive, inward negative.
- Absolute normal speed: magnitude only, to show where change is strongest.

Each subject is averaged across adjacent visits first, then the cohort mean map is shown.

In [17]:
for fig in helpers.local_shape_figures(no_mci.clean_df, 'ADNI original no-MCI', ['CN', 'AD']):
    fig.show()

for fig in helpers.local_shape_figures(with_mci.clean_df, 'ADNI original with-MCI', ['CN', 'MCI', 'AD']):
    fig.show()


## Step 3 Analysis

These tables quantify where inward shrinkage is strongest and whether the same hotspot area is shared across diagnoses.

- high hotspot overlap plus larger shared-hotspot inward speed means mostly the same area is affected, but faster;
- low hotspot overlap means the spatial pattern itself shifts.

The mesh itself does not contain hippocampal subfield labels, so this is a surface-hotspot analysis, not a direct `CA1` or `subiculum` annotation.

In [18]:
from IPython.display import display

no_hotspot = helpers.shape_hotspot_summary_table(no_mci.clean_df, ['CN', 'AD'], top_fraction=0.10, focus='inward')
no_similarity = helpers.pairwise_shape_similarity_table(no_mci.clean_df, ['CN', 'AD'], top_fraction=0.10, focus='inward')
with_hotspot = helpers.shape_hotspot_summary_table(with_mci.clean_df, ['CN', 'MCI', 'AD'], top_fraction=0.10, focus='inward')
with_similarity = helpers.pairwise_shape_similarity_table(with_mci.clean_df, ['CN', 'MCI', 'AD'], top_fraction=0.10, focus='inward')

print('No-MCI hotspot summary')
display(no_hotspot.round(5))
print('No-MCI pairwise hotspot similarity')
display(no_similarity.round(5))

print('With-MCI hotspot summary')
display(with_hotspot.round(5))
print('With-MCI pairwise hotspot similarity')
display(with_similarity.round(5))

for fig in helpers.shape_difference_figures(
    no_mci.clean_df,
    'ADNI original no-MCI',
    ['CN', 'AD'],
    [('CN', 'AD')],
):
    fig.show()

for fig in helpers.shape_difference_figures(
    with_mci.clean_df,
    'ADNI original with-MCI',
    ['CN', 'MCI', 'AD'],
    [('CN', 'MCI'), ('CN', 'AD'), ('MCI', 'AD')],
):
    fig.show()


No-MCI hotspot summary


,diagnosis,analysis_focus,subject_count,adjacent_pair_count,weighted_mean_focus_speed,weighted_mean_signed_speed,peak_focus_speed,hotspot_top_fraction,hotspot_vertex_count,hotspot_centroid_x,hotspot_centroid_y,hotspot_centroid_z
0,CN,inward,143,282,0.00076,-0.00041,0.00489,0.1,162,0.06438,0.11492,0.05890
1,AD,inward,101,201,0.00323,-0.00308,0.01070,0.1,160,0.02531,0.00381,0.04472


No-MCI pairwise hotspot similarity


,diagnosis_a,diagnosis_b,analysis_focus,top_fraction,focus_speed_corr,signed_speed_corr,hotspot_weighted_jaccard,hotspot_weighted_dice,hotspot_overlap_of_a,hotspot_overlap_of_b,shared_hotspot_mean_focus_speed_a,shared_hotspot_mean_focus_speed_b,shared_hotspot_speed_ratio_b_over_a
0,CN,AD,inward,0.1,0.40542,0.37398,0.27567,0.4322,0.43243,0.43197,0.003,0.00799,2.66137


With-MCI hotspot summary


,diagnosis,analysis_focus,subject_count,adjacent_pair_count,weighted_mean_focus_speed,weighted_mean_signed_speed,peak_focus_speed,hotspot_top_fraction,hotspot_vertex_count,hotspot_centroid_x,hotspot_centroid_y,hotspot_centroid_z
0,CN,inward,143,282,0.00076,-0.00041,0.00489,0.1,162,0.06438,0.11492,0.05890
1,MCI,inward,241,476,0.00168,-0.00154,0.00777,0.1,163,0.05927,0.10237,0.07355
2,AD,inward,101,201,0.00323,-0.00308,0.01070,0.1,160,0.02531,0.00381,0.04472


With-MCI pairwise hotspot similarity


,diagnosis_a,diagnosis_b,analysis_focus,top_fraction,focus_speed_corr,signed_speed_corr,hotspot_weighted_jaccard,hotspot_weighted_dice,hotspot_overlap_of_a,hotspot_overlap_of_b,shared_hotspot_mean_focus_speed_a,shared_hotspot_mean_focus_speed_b,shared_hotspot_speed_ratio_b_over_a
0,CN,MCI,inward,0.1,0.58511,0.56985,0.38604,0.55704,0.55747,0.55662,0.00318,0.00524,1.64687
1,CN,AD,inward,0.1,0.40542,0.37398,0.27567,0.43220,0.43243,0.43197,0.00300,0.00799,2.66137
2,MCI,AD,inward,0.1,0.67626,0.70161,0.35125,0.51989,0.52005,0.51972,0.00529,0.00802,1.51654


## Step 4: Literature-Guided Approximate Hippocampal Areas

These new cells do **not** claim true hippocampal subfields. The original mesh has no atlas labels, so the partition below is a coarse, literature-guided approximation.

What the partition tries to capture:

- an `anterior_head_like` sector;
- a `posterior_tail_like` sector;
- a `ca1_subiculum_like_band` in the middle portion;
- an `opposite_body_band` as the remaining middle flank.

Why this is only approximate:

- we use intrinsic mesh axes from PCA, not a manual anatomical atlas;
- PCA axis sign is arbitrary, so the vulnerable middle flank is oriented using the common AD literature prior that AD-related change often concentrates on a CA1/subiculum-side surface;
- normal aging localization is less consistent in the literature than AD, so `CN` is treated here mainly as the empirical aging reference inside this dataset.

Useful background reading:

- https://arxiv.org/abs/2007.04558
- https://arxiv.org/abs/2311.08176
- https://arxiv.org/abs/2302.00573


In [24]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

group_maps_lit = helpers.build_group_shape_maps(with_mci.clean_df, ['CN', 'MCI', 'AD'])
template_vertices_lit = np.mean(
    np.stack([group_maps_lit[d]['display_vertices'] for d in ['CN', 'MCI', 'AD']], axis=0),
    axis=0,
)
template_faces_lit = np.asarray(group_maps_lit['CN']['faces'], dtype=int)

# First intrinsic axis approximates head-tail. The second splits the two broad body flanks.
center_lit = template_vertices_lit.mean(axis=0, keepdims=True)
_, _, vh_lit = np.linalg.svd(template_vertices_lit - center_lit, full_matrices=False)
intrinsic_scores_lit = (template_vertices_lit - center_lit) @ vh_lit.T
ap_coord_lit = intrinsic_scores_lit[:, 0]
flank_coord_lit = intrinsic_scores_lit[:, 1]

head_threshold_lit = np.quantile(ap_coord_lit, 0.70)
tail_threshold_lit = np.quantile(ap_coord_lit, 0.25)
body_mask_lit = (ap_coord_lit > tail_threshold_lit) & (ap_coord_lit < head_threshold_lit)
flank_split_lit = np.median(flank_coord_lit[body_mask_lit])

# Literature-guided orientation: label the AD-more-vulnerable middle flank as CA1/subiculum-like.
ad_inward_lit = np.clip(-np.asarray(group_maps_lit['AD']['signed_speed'], dtype=float), 0.0, None)
body_high_mean_lit = float(ad_inward_lit[body_mask_lit & (flank_coord_lit >= flank_split_lit)].mean())
body_low_mean_lit = float(ad_inward_lit[body_mask_lit & (flank_coord_lit < flank_split_lit)].mean())
ca1_high_side_lit = body_high_mean_lit >= body_low_mean_lit

region_order_lit = [
    'anterior_head_like',
    'ca1_subiculum_like_band',
    'opposite_body_band',
    'posterior_tail_like',
]
region_palette_lit = {
    'anterior_head_like': '#d73027',
    'ca1_subiculum_like_band': '#fc8d59',
    'opposite_body_band': '#91bfdb',
    'posterior_tail_like': '#4575b4',
}
region_display_names_lit = {
    'anterior_head_like': 'Anterior head-like',
    'ca1_subiculum_like_band': 'CA1/subiculum-like band',
    'opposite_body_band': 'Opposite body band',
    'posterior_tail_like': 'Posterior tail-like',
}
region_labels_lit = np.empty(len(template_vertices_lit), dtype=object)
region_labels_lit[ap_coord_lit >= head_threshold_lit] = 'anterior_head_like'
region_labels_lit[ap_coord_lit <= tail_threshold_lit] = 'posterior_tail_like'
middle_mask_lit = (ap_coord_lit > tail_threshold_lit) & (ap_coord_lit < head_threshold_lit)
if ca1_high_side_lit:
    region_labels_lit[middle_mask_lit & (flank_coord_lit >= flank_split_lit)] = 'ca1_subiculum_like_band'
    region_labels_lit[middle_mask_lit & (flank_coord_lit < flank_split_lit)] = 'opposite_body_band'
else:
    region_labels_lit[middle_mask_lit & (flank_coord_lit < flank_split_lit)] = 'ca1_subiculum_like_band'
    region_labels_lit[middle_mask_lit & (flank_coord_lit >= flank_split_lit)] = 'opposite_body_band'

region_code_lit = {name: idx for idx, name in enumerate(region_order_lit)}
face_region_index_lit = np.array(
    [
        np.bincount(
            [region_code_lit[region_labels_lit[v]] for v in face_vertices_lit],
            minlength=len(region_order_lit),
        ).argmax()
        for face_vertices_lit in template_faces_lit
    ],
    dtype=int,
)
face_region_labels_lit = np.array([region_order_lit[idx] for idx in face_region_index_lit], dtype=object)
score_span_lit = intrinsic_scores_lit.max(axis=0) - intrinsic_scores_lit.min(axis=0)
base_camera_lit = dict(eye=dict(x=1.8, y=-1.8, z=0.9))


def _scores_to_xyz_lit(score_point_lit):
    score_point_lit = np.asarray(score_point_lit, dtype=float)
    return score_point_lit @ vh_lit + center_lit[0]


def _shape_layout_lit(title_lit, width_lit=900, height_lit=760):
    return dict(
        title=title_lit,
        template='plotly_white',
        width=width_lit,
        height=height_lit,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data',
            camera=base_camera_lit,
        ),
        margin=dict(l=0, r=0, t=60, b=0),
    )


def _add_region_mesh_traces_lit(fig_lit, scene_name_lit='scene', show_hover_lit=True, opacity_lit=1.0):
    for region_name_lit in region_order_lit:
        face_mask_lit = face_region_labels_lit == region_name_lit
        hovertemplate_lit = (
            f"{region_display_names_lit[region_name_lit]}<extra></extra>"
            if show_hover_lit
            else None
        )
        fig_lit.add_trace(
            go.Mesh3d(
                x=template_vertices_lit[:, 0],
                y=template_vertices_lit[:, 1],
                z=template_vertices_lit[:, 2],
                i=template_faces_lit[face_mask_lit, 0],
                j=template_faces_lit[face_mask_lit, 1],
                k=template_faces_lit[face_mask_lit, 2],
                color=region_palette_lit[region_name_lit],
                flatshading=True,
                opacity=opacity_lit,
                lighting=dict(ambient=0.5, diffuse=0.7, specular=0.15, roughness=0.8),
                name=region_display_names_lit[region_name_lit],
                legendgroup=region_name_lit,
                showlegend=False,
                hovertemplate=hovertemplate_lit,
                scene=scene_name_lit,
            )
        )


def _add_region_legend_traces_lit(fig_lit, scene_name_lit='scene'):
    for region_name_lit in region_order_lit:
        fig_lit.add_trace(
            go.Scatter3d(
                x=[None],
                y=[None],
                z=[None],
                mode='markers',
                marker=dict(size=10, color=region_palette_lit[region_name_lit]),
                name=region_display_names_lit[region_name_lit],
                legendgroup=region_name_lit,
                showlegend=True,
                hoverinfo='skip',
                scene=scene_name_lit,
            )
        )


def _region_anchor_and_label_points_lit():
    region_anchor_scores_lit = {
        'anterior_head_like': np.array([np.quantile(ap_coord_lit, 0.88), 0.02 * score_span_lit[1], 0.0]),
        'ca1_subiculum_like_band': np.array([
            intrinsic_scores_lit[region_labels_lit == 'ca1_subiculum_like_band', 0].mean(),
            intrinsic_scores_lit[region_labels_lit == 'ca1_subiculum_like_band', 1].mean(),
            intrinsic_scores_lit[region_labels_lit == 'ca1_subiculum_like_band', 2].mean(),
        ]),
        'opposite_body_band': np.array([
            intrinsic_scores_lit[region_labels_lit == 'opposite_body_band', 0].mean(),
            intrinsic_scores_lit[region_labels_lit == 'opposite_body_band', 1].mean(),
            intrinsic_scores_lit[region_labels_lit == 'opposite_body_band', 2].mean(),
        ]),
        'posterior_tail_like': np.array([np.quantile(ap_coord_lit, 0.08), -0.03 * score_span_lit[1], 0.0]),
    }
    label_offsets_lit = {
        'anterior_head_like': np.array([0.18, 0.30, 0.12]) * score_span_lit,
        'ca1_subiculum_like_band': np.array([0.03, 0.40, 0.10]) * score_span_lit,
        'opposite_body_band': np.array([-0.02, -0.40, -0.10]) * score_span_lit,
        'posterior_tail_like': np.array([-0.18, -0.28, 0.12]) * score_span_lit,
    }
    return {
        region_name_lit: (
            _scores_to_xyz_lit(region_anchor_scores_lit[region_name_lit]),
            _scores_to_xyz_lit(region_anchor_scores_lit[region_name_lit] + label_offsets_lit[region_name_lit]),
        )
        for region_name_lit in region_order_lit
    }


region_label_points_lit = _region_anchor_and_label_points_lit()


def make_region_atlas_figure_lit():
    fig_lit = go.Figure()
    _add_region_mesh_traces_lit(fig_lit, scene_name_lit='scene')
    _add_region_legend_traces_lit(fig_lit, scene_name_lit='scene')
    fig_lit.update_layout(**_shape_layout_lit('Approximate literature-guided hippocampal areas'))
    fig_lit.update_layout(
        legend=dict(x=1.02, y=1.0, bgcolor='rgba(255,255,255,0.9)'),
    )
    return fig_lit


def make_region_atlas_comparison_figure_lit():
    fig_lit = make_subplots(
        rows=1,
        cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=(
            'Interactive area-colored mesh',
            'Labeled still view with approximate area pointers',
        ),
        horizontal_spacing=0.02,
    )
    _add_region_mesh_traces_lit(fig_lit, scene_name_lit='scene')
    _add_region_legend_traces_lit(fig_lit, scene_name_lit='scene')
    _add_region_mesh_traces_lit(fig_lit, scene_name_lit='scene2', show_hover_lit=False)
    for region_name_lit in region_order_lit:
        anchor_point_lit, label_point_lit = region_label_points_lit[region_name_lit]
        fig_lit.add_trace(
            go.Scatter3d(
                x=[anchor_point_lit[0], label_point_lit[0]],
                y=[anchor_point_lit[1], label_point_lit[1]],
                z=[anchor_point_lit[2], label_point_lit[2]],
                mode='lines',
                line=dict(color=region_palette_lit[region_name_lit], width=6),
                showlegend=False,
                hoverinfo='skip',
                scene='scene2',
            )
        )
        fig_lit.add_trace(
            go.Scatter3d(
                x=[label_point_lit[0]],
                y=[label_point_lit[1]],
                z=[label_point_lit[2]],
                mode='markers+text',
                marker=dict(size=5, color=region_palette_lit[region_name_lit]),
                text=[region_display_names_lit[region_name_lit]],
                textposition='middle right',
                textfont=dict(size=12, color='black'),
                showlegend=False,
                hoverinfo='skip',
                scene='scene2',
            )
        )
    fig_lit.update_layout(
        title='Approximate literature-guided hippocampal areas: interactive mesh and labeled view',
        template='plotly_white',
        width=1450,
        height=760,
        margin=dict(l=0, r=0, t=80, b=0),
        legend=dict(x=0.42, y=0.98, bgcolor='rgba(255,255,255,0.9)'),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data',
            camera=base_camera_lit,
        ),
        scene2=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data',
            camera=base_camera_lit,
        ),
    )
    return fig_lit


def make_region_speed_table_lit():
    region_rows_lit = []
    for diagnosis_lit in ['CN', 'MCI', 'AD']:
        inward_lit = np.clip(-np.asarray(group_maps_lit[diagnosis_lit]['signed_speed'], dtype=float), 0.0, None)
        weights_lit = np.asarray(group_maps_lit[diagnosis_lit]['area_weights'], dtype=float)
        for region_name_lit in region_order_lit:
            region_mask_lit = region_labels_lit == region_name_lit
            region_rows_lit.append(
                {
                    'diagnosis': diagnosis_lit,
                    'region': region_name_lit,
                    'mean_inward_speed': float(np.average(inward_lit[region_mask_lit], weights=weights_lit[region_mask_lit])),
                    'peak_inward_speed': float(inward_lit[region_mask_lit].max()),
                    'area_fraction': float(weights_lit[region_mask_lit].sum() / weights_lit.sum()),
                }
            )
    return pd.DataFrame(region_rows_lit)


def make_regionwise_speed_shape_figure_lit(region_summary_lit, diagnosis_lit):
    region_speed_lookup_lit = dict(
        zip(
            region_summary_lit[region_summary_lit['diagnosis'] == diagnosis_lit]['region'],
            region_summary_lit[region_summary_lit['diagnosis'] == diagnosis_lit]['mean_inward_speed'],
        )
    )
    regionwise_intensity_lit = np.array([region_speed_lookup_lit[name] for name in region_labels_lit], dtype=float)
    fig_lit = go.Figure(
        data=[
            go.Mesh3d(
                x=template_vertices_lit[:, 0],
                y=template_vertices_lit[:, 1],
                z=template_vertices_lit[:, 2],
                i=template_faces_lit[:, 0],
                j=template_faces_lit[:, 1],
                k=template_faces_lit[:, 2],
                intensity=regionwise_intensity_lit,
                intensitymode='vertex',
                colorscale='YlOrRd',
                cmin=0.0,
                cmax=float(region_summary_lit['mean_inward_speed'].max()),
                colorbar=dict(title='region mean inward speed / year'),
                lighting=dict(ambient=0.45, diffuse=0.7, specular=0.2, roughness=0.7),
            )
        ]
    )
    fig_lit.update_layout(**_shape_layout_lit(f'{diagnosis_lit}: regionwise inward shrinkage on shape'))
    return fig_lit


In [25]:
from IPython.display import HTML, display

region_summary_lit = make_region_speed_table_lit()
region_pivot_lit = region_summary_lit.pivot(index='region', columns='diagnosis', values='mean_inward_speed').loc[region_order_lit, ['CN', 'MCI', 'AD']]
region_rank_lit = region_summary_lit.sort_values(['diagnosis', 'mean_inward_speed'], ascending=[True, False]).groupby('diagnosis').head(2).reset_index(drop=True)
region_ratio_lit = region_pivot_lit.copy()
region_ratio_lit['MCI_over_CN'] = region_ratio_lit['MCI'] / region_ratio_lit['CN']
region_ratio_lit['AD_over_CN'] = region_ratio_lit['AD'] / region_ratio_lit['CN']
region_ratio_lit['AD_over_MCI'] = region_ratio_lit['AD'] / region_ratio_lit['MCI']

legend_rows_lit = ''.join(
    [
        (
            f"<tr>"
            f"<td style='padding:6px 10px;border:1px solid #d9d9d9;'><div style='width:18px;height:18px;background:{region_palette_lit[r]};border:1px solid #333;'></div></td>"
            f"<td style='padding:6px 10px;border:1px solid #d9d9d9;'>{region_display_names_lit[r]}</td>"
            f"<td style='padding:6px 10px;border:1px solid #d9d9d9;'><code>{region_palette_lit[r]}</code></td>"
            f"</tr>"
        )
        for r in region_order_lit
    ]
)
display(
    HTML(
        "<div style='margin:0 0 8px 0; font-weight:600;'>Atlas color legend</div>"
        "<table style='border-collapse:collapse;'>"
        "<tr>"
        "<th style='padding:6px 10px;border:1px solid #d9d9d9;text-align:left;'>Color</th>"
        "<th style='padding:6px 10px;border:1px solid #d9d9d9;text-align:left;'>Approximate area</th>"
        "<th style='padding:6px 10px;border:1px solid #d9d9d9;text-align:left;'>Hex</th>"
        "</tr>"
        f"{legend_rows_lit}"
        "</table>"
    )
)
print('Side-by-side atlas: left is the interactive categorical mesh, right is the labeled still view.')
make_region_atlas_comparison_figure_lit().show()
print('Top two approximate literature-guided areas per diagnosis, ranked by inward shrinkage speed')
display(region_rank_lit.round(6))
print('Mean inward shrinkage speed by approximate area and diagnosis')
display(region_pivot_lit.round(6))
print('Speed ratios by approximate area')
display(region_ratio_lit.round(6))

fig_region_heat_lit = go.Figure(
    data=[
        go.Heatmap(
            z=region_pivot_lit.values,
            x=list(region_pivot_lit.columns),
            y=list(region_pivot_lit.index),
            colorscale='YlOrRd',
            colorbar=dict(title='mean inward speed / year'),
        )
    ]
)
fig_region_heat_lit.update_layout(
    title='Approximate literature-guided area by diagnosis: mean inward shrinkage speed',
    template='plotly_white',
    width=900,
    height=450,
)

fig_region_heat_lit.show()
for diagnosis_lit in ['CN', 'MCI', 'AD']:
    make_regionwise_speed_shape_figure_lit(region_summary_lit, diagnosis_lit).show()


Color,Approximate area,Hex
,Anterior head-like,#d73027
,CA1/subiculum-like band,#fc8d59
,Opposite body band,#91bfdb
,Posterior tail-like,#4575b4


Side-by-side atlas: left is the interactive categorical mesh, right is the labeled still view.


Top two approximate literature-guided areas per diagnosis, ranked by inward shrinkage speed


,diagnosis,region,mean_inward_speed,peak_inward_speed,area_fraction
0,AD,anterior_head_like,0.004059,0.010696,0.305907
1,AD,posterior_tail_like,0.003436,0.009274,0.248544
2,CN,posterior_tail_like,0.001074,0.003896,0.261751
3,CN,anterior_head_like,0.000707,0.004892,0.306130
4,MCI,posterior_tail_like,0.002268,0.007768,0.256393
5,MCI,anterior_head_like,0.001576,0.005531,0.306253


Mean inward shrinkage speed by approximate area and diagnosis


diagnosis,CN,MCI,AD
region,,,
anterior_head_like,0.000707,0.001576,0.004059
ca1_subiculum_like_band,0.000662,0.001413,0.002963
opposite_body_band,0.000528,0.001408,0.002100
posterior_tail_like,0.001074,0.002268,0.003436


Speed ratios by approximate area


diagnosis,CN,MCI,AD,MCI_over_CN,AD_over_CN,AD_over_MCI
region,,,,,,
anterior_head_like,0.000707,0.001576,0.004059,2.229034,5.740316,2.575248
ca1_subiculum_like_band,0.000662,0.001413,0.002963,2.135090,4.477924,2.097299
opposite_body_band,0.000528,0.001408,0.002100,2.667032,3.975904,1.490760
posterior_tail_like,0.001074,0.002268,0.003436,2.111205,3.198295,1.514915


## Step 4 Legend

Area names and colors used for the approximate literature-guided hippocampal partition.